# Build Electricity Load Dataset

Notebook version of the ENTSO-E + Open-Meteo dataset builder.

The ENTSO-E API key is read from `.env`:

```text
ENTSOE_API_KEY=your_token_here
```

In [ ]:
from pathlib import Path
import os

import holidays
import openmeteo_requests
import pandas as pd
import requests_cache
from dotenv import load_dotenv
from entsoe import EntsoePandasClient
from retry_requests import retry

pd.set_option("display.max_columns", 100)

## 1. Settings

Change these values if you want a different period, country/bidding zone, or weather location.

In [ ]:
PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# ENTSO-E bidding zone. Default is Germany/Luxembourg.
COUNTRY_CODE = "DE_LU"

# Inclusive date range.
START_DATE = "2023-01-01"
END_DATE = "2023-10-31"

TIMEZONE = "Europe/Berlin"

# Weather point. Default is Berlin.
LATITUDE = 52.52
LONGITUDE = 13.41

load_dotenv(PROJECT_ROOT / ".env")
ENTSOE_API_KEY = os.getenv("ENTSOE_API_KEY")

if not ENTSOE_API_KEY:
    raise RuntimeError("ENTSOE_API_KEY was not found. Add it to .env first.")

start = pd.Timestamp(START_DATE, tz=TIMEZONE)
end = pd.Timestamp(END_DATE, tz=TIMEZONE) + pd.Timedelta(days=1)

start, end

## 2. Helper Functions

In [ ]:
def month_windows(start_ts: pd.Timestamp, end_ts: pd.Timestamp):
    windows = []
    cursor = start_ts
    while cursor < end_ts:
        next_month = (cursor + pd.offsets.MonthBegin(1)).normalize()
        window_end = min(next_month, end_ts)
        windows.append((cursor, window_end))
        cursor = window_end
    return windows


def normalize_entsoe_result(result, name: str, timezone: str) -> pd.DataFrame:
    if isinstance(result, pd.Series):
        frame = result.rename(name).to_frame()
    elif isinstance(result, pd.DataFrame):
        numeric_columns = result.select_dtypes(include="number").columns.tolist()
        if len(numeric_columns) == 1:
            frame = result[[numeric_columns[0]]].rename(columns={numeric_columns[0]: name})
        elif name in result.columns:
            frame = result[[name]]
        else:
            frame = result.iloc[:, [0]].rename(columns={result.columns[0]: name})
    else:
        raise TypeError(f"Unsupported ENTSO-E result type: {type(result)}")

    frame.index = pd.to_datetime(frame.index)
    if frame.index.tz is None:
        frame.index = frame.index.tz_localize(timezone)
    else:
        frame.index = frame.index.tz_convert(timezone)
    frame[name] = pd.to_numeric(frame[name], errors="coerce")
    return frame.sort_index()


def fetch_entsoe_load(api_key: str, country_code: str, start_ts: pd.Timestamp, end_ts: pd.Timestamp, timezone: str):
    client = EntsoePandasClient(api_key=api_key)
    actual_parts = []
    forecast_parts = []

    for window_start, window_end in month_windows(start_ts, end_ts):
        print(f"Fetching ENTSO-E {window_start.date()} -> {(window_end - pd.Timedelta(days=1)).date()}")
        actual = client.query_load(country_code, start=window_start, end=window_end)
        forecast = client.query_load_forecast(country_code, start=window_start, end=window_end)
        actual_parts.append(normalize_entsoe_result(actual, "load_mw", timezone))
        forecast_parts.append(normalize_entsoe_result(forecast, "load_forecast_mw", timezone))

    actual_load = pd.concat(actual_parts)
    forecast_load = pd.concat(forecast_parts)
    actual_load = actual_load.loc[~actual_load.index.duplicated(keep="first")]
    forecast_load = forecast_load.loc[~forecast_load.index.duplicated(keep="first")]
    return actual_load.join(forecast_load, how="outer").sort_index()


def fetch_open_meteo(start_ts: pd.Timestamp, end_ts: pd.Timestamp, latitude: float, longitude: float, timezone: str):
    cache_session = requests_cache.CachedSession(str(PROJECT_ROOT / ".cache" / "openmeteo"), expire_after=-1)
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
    client = openmeteo_requests.Client(session=retry_session)

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_ts.date().isoformat(),
        "end_date": (end_ts - pd.Timedelta(days=1)).date().isoformat(),
        "hourly": ["temperature_2m", "relative_humidity_2m", "wind_speed_10m"],
        "timezone": timezone,
    }

    print("Fetching Open-Meteo weather")
    response = client.weather_api("https://archive-api.open-meteo.com/v1/archive", params=params)[0]
    hourly = response.Hourly()

    index = pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit="s", utc=True).tz_convert(timezone),
        end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True).tz_convert(timezone),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive="left",
    )

    return pd.DataFrame(
        {
            "temperature_2m": hourly.Variables(0).ValuesAsNumpy(),
            "relative_humidity_2m": hourly.Variables(1).ValuesAsNumpy(),
            "wind_speed_10m": hourly.Variables(2).ValuesAsNumpy(),
        },
        index=index,
    )


def add_features(frame: pd.DataFrame, country_code: str):
    dataset = frame.copy()
    dataset["hour"] = dataset.index.hour
    dataset["dayofweek"] = dataset.index.dayofweek
    dataset["month"] = dataset.index.month
    dataset["is_weekend"] = dataset["dayofweek"].isin([5, 6]).astype(int)

    holiday_country = country_code.split("_")[0]
    holiday_dates = holidays.country_holidays(holiday_country)
    dataset["is_holiday"] = [int(ts.date() in holiday_dates) for ts in dataset.index]

    dataset["load_lag_24h"] = dataset["load_mw"].shift(24)
    dataset["load_lag_168h"] = dataset["load_mw"].shift(168)
    dataset["load_rolling_24h_mean"] = dataset["load_mw"].shift(1).rolling(24).mean()
    return dataset

## 3. Fetch ENTSO-E Load Data

ENTSO-E data often arrives at 15-minute granularity. The final model dataset below resamples it to hourly averages.

In [ ]:
load_frame = fetch_entsoe_load(ENTSOE_API_KEY, COUNTRY_CODE, start, end, TIMEZONE)
load_frame.to_csv(RAW_DIR / "entsoe_load.csv", index_label="timestamp")

print(load_frame.shape)
display(load_frame.head())
display(load_frame.tail())

## 4. Fetch Weather Data

In [ ]:
weather_frame = fetch_open_meteo(start, end, LATITUDE, LONGITUDE, TIMEZONE)
weather_frame.to_csv(RAW_DIR / "open_meteo_weather.csv", index_label="timestamp")

print(weather_frame.shape)
display(weather_frame.head())
display(weather_frame.tail())

## 5. Build Final Hourly Dataset

In [ ]:
hourly_load = load_frame.resample("h").mean()
hourly_weather = weather_frame.resample("h").mean()

dataset = hourly_load.join(hourly_weather, how="left")
dataset = add_features(dataset, COUNTRY_CODE)
dataset = dataset.dropna(subset=["load_mw"]).sort_index()

output_path = PROCESSED_DIR / "power_load_dataset.csv"
dataset.to_csv(output_path, index_label="timestamp")

print("saved:", output_path)
print(dataset.shape)
display(dataset.head())
display(dataset.tail())

## 6. Quick Quality Checks

In [ ]:
missing = dataset.isna().sum().sort_values(ascending=False)
display(missing[missing > 0])

deltas = dataset.index.to_series().diff().dropna()
most_common_interval = deltas.mode().iloc[0] if not deltas.empty else None
irregular_steps = int((deltas != most_common_interval).sum()) if most_common_interval is not None else 0

print("time start:", dataset.index.min())
print("time end:", dataset.index.max())
print("most common interval:", most_common_interval)
print("irregular steps:", irregular_steps)

## 7. Optional: Verify Hourly Aggregation

This shows why four 15-minute values become one hourly value: the hourly value is the mean of the four rows.

In [ ]:
first_hour = load_frame.loc["2023-01-01 00:00:00+01:00":"2023-01-01 00:45:00+01:00"]
display(first_hour)

print("15-minute mean load_mw:", first_hour["load_mw"].mean())
print("hourly dataset load_mw:", dataset.loc[pd.Timestamp("2023-01-01 00:00:00", tz=TIMEZONE), "load_mw"])